# Dirichlet Regression

In [ ]:
library("DirichletReg")
library(tibble)

## Confs

In [ ]:
WORKING_DIR <- Sys.getenv("REPO_ROOT", unset = normalizePath(".."))
DATA_DIR <- paste(WORKING_DIR, '/data', sep='')

PREDICTION_DIR_2019 <- paste(DATA_DIR, '/dirichlet/2019', sep='')
PREDICTION_DIR_2024 <- paste(DATA_DIR, '/dirichlet/2024', sep='')

### Load election results + social economic indicators

In [ ]:
filename_election_2019 <- paste(DATA_DIR, '/dirichlet/df_data_europe_2019_selected.csv', sep='')
df_election_social_traffic_2019 = read.csv(filename_election_2019, sep=',', header=TRUE)

filename_election_2024 <- paste(DATA_DIR, '/dirichlet/df_data_europe_2024_selected.csv', sep='')
df_election_social_traffic_2024 = read.csv(filename_election_2024, sep=',', header=TRUE)

year <- 2019
year <- 2024


# head(df_election_social_traffic, 2)
# n_parties


if (year == 2019) {
    df_election_social_traffic <- df_election_social_traffic_2019
    n_parties <- 5
    apps_columns = colnames(df_election_social_traffic)[22:length(colnames(df_election_social_traffic))]
} else {
    df_election_social_traffic <- df_election_social_traffic_2024
    n_parties <- 6
    apps_columns = colnames(df_election_social_traffic)[23:length(colnames(df_election_social_traffic))]
}

head(df_election_social_traffic, 2)
n_parties

In [ ]:
apps_columns

## Dirichlet Regression

### Select parties votes

In [ ]:
df_party_votes = DR_data(df_election_social_traffic[, 2:(2+n_parties+1)])
head(df_party_votes, 2)

### Regression constant values

In [ ]:
regression_intercept <- DirichReg(df_party_votes ~ 1, 
                        df_election_social_traffic,
                        control = list(iterlim = 1000, tol1 = 1e-5))

summary(regression_intercept)

### Regression using social economic indicators

#### Pop

In [ ]:
regression_pop <- DirichReg(df_party_votes ~ pop_0_14 +
                                            pop_15_29 +
                                            pop_30_44 +
                                            pop_45_59 +
                                            pop_60_74, # removing pop_75_89 and pop_90 to avoid multicollinearity
                        df_election_social_traffic,
                        control = list(iterlim = 1000, tol1 = 1e-5))

summary(regression_pop)

#### Unemployment

In [ ]:
regression_unemployment <- DirichReg(df_party_votes ~ unemployment_ratio, 
                        df_election_social_traffic, 
                        control = list(iterlim = 1000, tol1 = 1e-5))

summary(regression_unemployment)

#### Income

In [ ]:
regression_income <- DirichReg(df_party_votes ~ median_income,
                        df_election_social_traffic,
                        control = list(iterlim = 1000, tol1 = 1e-5))
summary(regression_income)

#### Income + Unemployment + Pop

In [ ]:
social_columns = c('median_income', 'unemployment_ratio',
                   'pop_0_14', 'pop_15_29', 'pop_30_44', 'pop_45_59', 'pop_60_74') # removing pop_75_89 and pop_90 to avoid multicollinearity

str_formula <- paste('df_party_votes ~ ', paste(social_columns, collapse = ' + '), sep='')

regression_income_unemployment_pop <- DirichReg(
                        formula = as.formula(str_formula),
                        df_election_social_traffic,
                        control = list(iterlim = 1000, tol1 = 1e-5))
                        
summary(regression_income_unemployment_pop)

#### Traffic

##### Social Media

In [ ]:
if (year == 2019) {
    social_media_columns <- c(
        'Facebook_srca',
        'Instagram_srca',
        'LinkedIn_srca',
        'SnapChat_srca',
        'Twitter_srca',
        'Twitch_srca'
    )
} else {
    social_media_columns <- c(
        'Facebook_srca',
        'Instagram_srca',
        'LinkedIn_srca',
        'SnapChat_srca',
        'Twitter_srca',
        'Twitch_srca',
        'TikTok_srca'
    )
}

social_media_columns

In [ ]:
str_formula <- paste('df_party_votes ~ ', paste(social_media_columns, collapse = ' + '), sep='')

regression_social_media <- DirichReg( formula = as.formula(str_formula),
                        df_election_social_traffic,
                        control = list(iterlim = 1000, tol1 = 1e-5))
                        
summary(regression_social_media)

##### News

In [ ]:
news_columns = c(
    'NewsPaper_srca',
    'Sports.News_srca',
    'DailyMotion_srca',
    'NewsMag_srca',
    'Google.News_srca',
    'TV5MONDE_srca')

str_formula <- paste('df_party_votes ~ ', paste(news_columns, collapse = ' + '), sep='')

regression_news <- DirichReg( formula = as.formula(str_formula),
                        df_election_social_traffic,
                        control = list(iterlim = 1000, tol1 = 1e-5))
                        
summary(regression_news)

##### Messaging

In [ ]:
if (year == 2019) {
    messaging_columns = c(
        'WhatsApp_srca',
        'Apple.iMessage_srca',
        'Telegram_srca'
    )
} else {
    messaging_columns = c(
        'WhatsApp_srca',
        'Apple.iMessage_srca',
        'Signal_srca',
        'Discord_srca',
        'Telegram_srca')
}

str_formula <- paste('df_party_votes ~ ', paste(messaging_columns, collapse = ' + '), sep='')

regression_messaging <- DirichReg( formula = as.formula(str_formula),
                        df_election_social_traffic,
                        control = list(iterlim = 1000, tol1 = 1e-5))
                        
summary(regression_messaging)

##### Streamming

In [ ]:
if (year == 2019) {
    streamming_columns = c(
        'Youtube_srca',
        'Spotify_srca',
        'CanalPlus_srca',
        'Netflix_srca',
        'Apple.Music_srca',
        'Apple.Video_srca',
        'Molotov.TV_srca'
    )
} else {
streamming_columns = c(
    'Youtube_srca',
    'Spotify_srca',
    'CanalPlus_srca',
    'Netflix_srca',
    'Apple.Music_srca',
    'Disney._srca',
    'Apple.Video_srca',
    'Molotov.TV_srca',
    'Pluto.TV_srca'
)
}


str_formula <- paste('df_party_votes ~ ', paste(streamming_columns, collapse = ' + '), sep='')

regression_streamming <- DirichReg( formula = as.formula(str_formula),
                        df_election_social_traffic,
                        control = list(iterlim = 1000, tol1 = 1e-5))
                        
summary(regression_streamming)

##### Apps

In [ ]:
apps_columns

In [ ]:
str_formula <- paste('df_party_votes ~ ', paste(apps_columns, collapse = ' + '), sep='')

regression_apps <- DirichReg( formula = as.formula(str_formula),
                        df_election_social_traffic,
                        control = list(iterlim = 1000, tol1 = 1e-5))
                        
summary(regression_apps)

#### Income + Unemployment + Pop + Apps

In [ ]:
social_traffic_columns = c(social_columns, apps_columns)
str_formula <- paste('df_party_votes ~ ', paste(social_traffic_columns, collapse = ' + '), sep='')

regression_income_unemployment_pop_apps <- DirichReg(formula = as.formula(str_formula),
                                            df_election_social_traffic,
                                            control = list(iterlim = 1000, tol1 = 1e-5))

summary(regression_income_unemployment_pop_apps)

## Models Performance Evaluation

In [ ]:
model_names <- list('intercept', 'pop', 'unemployment', 'income', 
                'apps', 
                'income_unemployment_pop',
                'social_media',
                'news',
                'messaging',
                'streamming',
                'income_unemployment_pop_apps')

models <- list( regression_intercept, regression_pop, regression_unemployment, regression_income, 
                regression_apps, 
                regression_income_unemployment_pop,
                regression_social_media,
                regression_news,
                regression_messaging,
                regression_streamming,
                regression_income_unemployment_pop_apps)


list_logML_model <- list()
list_n_parameters <- list()
list_aic <- list()
list_bic <- list()

for (i in 1:length(model_names)){

    model_name <- model_names[i]
    model <- models[[i]]
    coefficients <- model['coefficients']
    std_errors <- sqrt(diag(vcov(model)))

    n_parameter <- model['npar']
    logML_model <- round(logLik(model)[1], 2)
    aic_model   <- round(AIC(model), 2)
    bic_model   <- round(BIC(model), 2)

    len <- length(list_logML_model)
    list_logML_model[[len+1]] <- logML_model
    list_n_parameters[[len+1]] <- n_parameter
    list_aic[[len+1]] <- aic_model
    list_bic[[len+1]] <- bic_model

    print(paste('Predicting', model_name, '...'))
    prediction <- predict(model)

    filename_model <- paste(DATA_DIR, '/dirichlet/', year , '/df_prediction_', model_name, '.csv', sep='')
    print(paste('Saving prediction to', filename_model, '...'))
    write.csv(prediction, filename_model, row.names=FALSE)

    filename_model_coefficients <- paste(DATA_DIR, '/dirichlet/', year, '/df_coefficients_', model_name, '.csv', sep='')
    write.csv(coefficients, filename_model_coefficients)

    filename_model_std_error <- paste(DATA_DIR, '/dirichlet/', year, '/df_coefficients_std_error_', model_name, '.csv', sep='')
    write.csv(std_errors, filename_model_std_error)
}

df_model_parameters <- data.frame(
                    'model' = unlist(model_names),
                    'n_parameters' = unlist(list_n_parameters),
                    'LogML' = unlist(list_logML_model),
                    'AIC'   = unlist(list_aic),
                    'BIC'   = unlist(list_bic))

filename_parameters <- paste(DATA_DIR, '/dirichlet/', year, '/df_parameters_fit.csv', sep='')
write.csv(df_model_parameters, filename_parameters, row.names=FALSE)
head(df_model_parameters, 12)